# Validación MLOFI vs OFI L1 + serie descriptiva de VPIN

Fuente de verdad para el OLS: `scripts/validate_mlofi.py` (misma captura y misma
rejilla 1s/5s/10s que la fase 4). Este notebook lo ejecuta y pinta VPIN como
**descriptor retrospectivo**, no como alarma.

- MLOFI: [docs/math/mlofi.md](../docs/math/mlofi.md), informe
  [mlofi_validation.md](../docs/math/mlofi_validation.md).
- VPIN: [docs/math/vpin.md](../docs/math/vpin.md). Andersen & Bondarenko (2014):
  los máximos en el Flash Crash llegaron **después** del colapso.

```bash
uv run --extra notebooks python scripts/validate_mlofi.py --root data/live-btcusdt-45min
```


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
repo = ROOT if (ROOT / "scripts" / "validate_mlofi.py").is_file() else ROOT.parent
sys.path.insert(0, str(repo / "scripts"))
import validate_mlofi  # noqa: E402

capture = repo / "data" / "live-btcusdt-45min"
if not capture.is_dir():
    capture = repo / "data" / "live-btcusdt-5min"
report = repo / "docs" / "math" / "mlofi_validation.md"
print("capture", capture)
rc = validate_mlofi.main(["--root", str(capture), "--report", str(report)])
print("exit", rc)
print(report.read_text(encoding="utf-8")[:5000] if report.is_file() else "no report")

## VPIN descriptivo (no es un test de predicción de crash)

`compute_retrospective_vpin` / `vpin_from_capture`: buckets de volumen fijo $V$,
no una ventana de tiempo. $V$ se elige para tener ~200 buckets en esta muestra
corta (el paper usa ADV/50 en un día entero; aquí no hay un día). Ventana
rolling $n=50$ buckets. Clasificación por agresor Binance (`m`).


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

from order_flow.ingestion.binance_futures import EXCHANGE
from order_flow.metrics.batch import vpin_from_capture
from order_flow.metrics.vpin import bucket_trades, compute_retrospective_vpin
from order_flow.storage.parquet import read_events, trades_from_frame

frame = read_events(capture, "trade", exchange=EXCHANGE, symbol="BTCUSDT")
trades = trades_from_frame(frame)
price = np.array([t.price for t in trades], dtype=np.float64)
qty = np.array([t.qty for t in trades], dtype=np.float64)
signs = np.array([t.aggressor.sign for t in trades], dtype=np.float64)
ts = np.array([t.ts_event_ns for t in trades], dtype=np.int64)

total_vol = float(qty.sum())
n_target = 200
bucket_size = max(total_vol / n_target, 1e-9)
window = 50
vpin = compute_retrospective_vpin(price, qty, bucket_size, window, aggressor=signs)
batch = vpin_from_capture(
    capture,
    exchange=EXCHANGE,
    symbol="BTCUSDT",
    bucket_size=bucket_size,
    window=window,
)
np.testing.assert_allclose(vpin, batch)

buckets = bucket_trades(price, qty, bucket_size, signs)
cum = np.cumsum(qty)
boundaries = bucket_size * np.arange(1, buckets.n_buckets + 1)
idx = np.searchsorted(cum, boundaries * (1.0 - 1e-9), side="left")
idx = np.minimum(idx, ts.shape[0] - 1)
t_sec = (ts[idx] - ts[0]) / 1e9
t_vpin = t_sec[window - 1 :]

print(
    f"trades={len(trades)} total_vol={total_vol:.4f} V={bucket_size:.4f} "
    f"n_buckets={buckets.n_buckets} n_vpin={vpin.size} "
    f"min={vpin.min() if vpin.size else math.nan:.3f} "
    f"max={vpin.max() if vpin.size else math.nan:.3f}"
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_vpin, vpin, lw=1.0, color="#1f4e79")
ax.set_ylim(0.0, 1.0)
ax.set_xlabel("seconds from first trade")
ax.set_ylabel("retrospective VPIN")
ax.set_title("VPIN (aggressor, volume clock) — descriptive, not an early-warning test")
ax.grid(True, alpha=0.3)
fig.tight_layout()